In [1]:
import numpy as np
import pandas as pd

In [2]:
from google.colab import drive

drive.mount('/content/drive')

import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/ML_Datasets/imdb-dataset.csv')

Mounted at /content/drive


In [3]:
# df = pd.read_csv("imdb-dataset.csv")

In [4]:
df.shape

(50000, 2)

In [5]:
df.drop_duplicates(inplace = True)

In [6]:
df.shape

(49582, 2)

In [7]:
# CONVERTING IN LOVERWCASE
df['review'] = df['review'].str.lower()

In [8]:
# REMOVING LINKS
import re

def remove_urls(text):
  text = re.sub(r"http\S+", "", text)
  return text

df['review'] = df['review'].apply(remove_urls)

In [9]:
# REMOVE PUNCTUATIONS
def remove_punctuations(text):

  text = re.sub(r"[^A-Za-z0-9\s]", "", text)
  return text

df['review'] = df['review'].apply(remove_punctuations)

In [10]:

def remove_html(text):
    text = re.sub(r"<.*?>" , "", text)
    return text

df["review"] = df["review"].apply(remove_html)

In [11]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [12]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [13]:
def remove_stopwords(text):
  tokens = word_tokenize(text)
  stop_words = stopwords.words("english")

  for word in tokens:
    if word in stop_words:
      text = text.replace(word, "")

  return text

df['review'] = df['review'].apply(remove_stopwords)

In [14]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


In [15]:
# importing the porter stemer to remove grammer
from nltk.stem import PorterStemmer

In [16]:

def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [17]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


In [18]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df['sentiment'] = le.fit_transform(df['sentiment'])

In [19]:
y = df['sentiment']

In [20]:
y

,sentiment
0,1
1,1
2,1
3,0
4,1
...,...
49995,1
49996,0
49997,0
49998,0


In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features = 5000)

X = tf.fit_transform(df['review'])

In [22]:
# Dataset and detaloader

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42
)

In [23]:
X_train.shape

(39665, 5000)

In [24]:
X_test.shape

(9917, 5000)

In [25]:
import torch
from torch.utils.data import TensorDataset, DataLoader



In [26]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [ ]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [28]:
train_loader = DataLoader(train_set, shuffle = True, batch_size = 64,)
test_loader = DataLoader(test_set, shuffle = True, batch_size = 64)

In [29]:
import torch.nn as nn
import torch.optim as optim

In [30]:
class RNN(nn.Module):
  def __init__(self, input_size, hidden_size = 128, num_layers = 1):
    super().__init__()

    self.hidden_size = hidden_size
    self.num_layers = num_layers

    # RNN layers
    self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first = True)

    # FUlly_connected layers
    self.fc = nn.Linear(hidden_size, 1)

  def forward(self, x):
    h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

    out, _ = self.rnn(x, h0)

    out = self.fc(out[:, -1, :])
    return out

In [31]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()

optimizer = optim.Adam(model.parameters())

In [34]:
epochs = 12

for epoch in range(epochs):
  model.train()

  for Xb, yb in train_loader:
    optimizer.zero_grad()

    Xb = Xb.unsqueeze(1)

    outputs = model(Xb)

    outputs = torch.sigmoid(outputs.squeeze())

    loss = criterion(outputs, yb)
    loss.backward()
    optimizer.step()

  print(f"epoch = {epoch + 1}/{epochs} and loss = {loss.item()}")

epoch = 1/12 and loss = 0.2015320360660553
epoch = 2/12 and loss = 0.14015521109104156
epoch = 3/12 and loss = 0.2328488677740097
epoch = 4/12 and loss = 0.11525768786668777
epoch = 5/12 and loss = 0.338871031999588
epoch = 6/12 and loss = 0.2252987027168274
epoch = 7/12 and loss = 0.18651235103607178
epoch = 8/12 and loss = 0.11407935619354248
epoch = 9/12 and loss = 0.28889310359954834
epoch = 10/12 and loss = 0.3632497191429138
epoch = 11/12 and loss = 0.24437886476516724
epoch = 12/12 and loss = 0.25969308614730835


In [35]:
model.eval()

with torch.no_grad():
  correct_vals = 0
  tot_vals = 0

  for Xb, yb in test_loader:
    Xb = Xb.unsqueeze(1)

    outputs = model(Xb)

    predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

    tot_vals += yb.size(0)
    correct_vals += (predicted == yb).sum().item()

  print(f"Accuracy = {correct_vals / tot_vals * 100}")

Accuracy = 85.33830795603508
